# 🚀 End-to-End RAG Pipeline & RAG Triad Evaluation with LangSmith

**Production Masterclass by Krish Naik | KRISHAI Technologies & Production Engineering**

Retrieval-Augmented Generation (RAG) is the foundational architecture for enterprise AI applications. However, deploying a RAG pipeline to production without rigorous evaluation is dangerous: retrievers can fetch irrelevant documents, generators can hallucinate facts, and answers can drift from the user's intent.

The industry-standard framework for evaluating RAG systems is the **RAG Triad**:
1. **Context Relevance (Retrieval Quality)**: Did the retriever fetch chunks directly related to the user's query?
2. **Groundedness / Faithfulness (Hallucination Detection)**: Is the generated answer 100% justified by the retrieved context, or did the model invent facts?
3. **Answer Relevance**: Did the model actually answer what the user asked?
4. **Factual Correctness**: Does the response agree with verified human ground truth?

---

![RAG Evaluation and RAG Triad Architecture](workflow_rag_evaluation.png)

<details>
<summary><b>🔍 Click to Expand Interactive Mermaid Architecture Diagram</b></summary>

```mermaid
flowchart TD
    classDef ingest fill:#EBF5FB,stroke:#2980B9,stroke-width:2px,color:#1B4F72;
    classDef store fill:#D5F5E3,stroke:#27AE60,stroke-width:2px,color:#145A32;
    classDef gen fill:#F4ECF7,stroke:#8E44AD,stroke-width:2px,color:#512E5F;
    classDef triad fill:#FEF9E7,stroke:#F39C12,stroke-width:2px,color:#7D6608;
    classDef eval fill:#FDEDEC,stroke:#E74C3C,stroke-width:2px,color:#78281F;

    Docs["📚 Curated Knowledge Base<br/>(AI Agents, Prompting, Adversarial Attacks)"]:::ingest
    Docs --> Split["✂️ Recursive Character Chunker<br/>(Chunk size: 300, Overlap: 30)"]:::ingest
    Split --> Emb["🔢 HuggingFace MiniLM Embeddings<br/>(all-MiniLM-L6-v2 - 384 Dim)"]:::store
    Emb --> VStore[("💾 InMemoryVectorStore<br/>(Dense Cosine Similarity Search)")]:::store

    UserQ["❓ User Test Question"]:::ingest --> Ret["🔍 Vector Retriever (Top-k=2)"]:::store
    VStore --> Ret
    Ret --> Context["📄 Retrieved Context Chunks"]:::store

    UserQ --> LLM["⚡ Groq High-Speed Generator<br/>(qwen/qwen3.8-27b)"]:::gen
    Context --> LLM
    LLM --> Ans["💬 Generated Grounded Answer"]:::gen

    subgraph Triad ["⚖️ The RAG Triad & Benchmark Evaluators"]
        M1["1️⃣ Retrieval Relevance<br/>(Query vs Retrieved Chunks)"]:::triad
        M2["2️⃣ Groundedness / Faithfulness<br/>(Retrieved Chunks vs Answer)"]:::triad
        M3["3️⃣ Answer Relevance<br/>(Query vs Final Answer)"]:::triad
        M4["4️⃣ Ground Truth Correctness<br/>(Final Answer vs Reference)"]:::triad
    end

    UserQ --> M1
    Context --> M1

    Context --> M2
    Ans --> M2

    UserQ --> M3
    Ans --> M3

    RefAns["🎯 Ground Truth Reference"]:::ingest --> M4
    Ans --> M4

    Triad --> LS["🌐 LangSmith Observability & Scorecard<br/>(Run Traces, Pass Rates & Latency Analytics)"]:::eval
```

</details>

---

### 📚 What You'll Master
1. **The RAG Triad Architecture**: Why evaluating RAG requires decoupling retrieval accuracy from generative faithfulness.
2. **Dense Semantic Indexing**: Building an in-memory vector index with HuggingFace MiniLM (`all-MiniLM-L6-v2`) dense vectors.
3. **Traceable RAG Workflows**: Instrumenting RAG pipelines using `@traceable()` to capture every intermediate context chunk in LangSmith.
4. **Context Relevance Evaluator**: Automated grading of whether retrieved chunks contain actionable signal or distracting noise.
5. **Groundedness / Hallucination Evaluator**: Rigorous verification that every claim in the response is backed by retrieved evidence.
6. **Answer Relevance & Correctness**: Ensuring user intent is directly fulfilled and factual claims match gold-standard truth.
7. **End-to-End LangSmith Evaluation**: Executing automated test suites with `client.evaluate` and analyzing multi-metric Pandas scorecards.


## ⚙️ Section 1: Environment Setup & Verifications

We load our environment variables, verify connectivity to **Groq** for high-throughput generation and evaluation, and establish a connection to **LangSmith** for trace tracking and benchmark storage.


In [1]:
"""
In this initial step, we set up our workspace and ensure our environment is healthy.
We load our API credentials from the local .env file so our RAG application can call Groq's
lightning-fast models and send benchmark metrics straight to our LangSmith dashboard.
We also silence noisy background logging to keep our outputs clear and pleasant to read.
"""

import os
import sys
import time
import json
import warnings
import logging
import pandas as pd
from dotenv import load_dotenv

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
langsmith_key = os.getenv("LANGSMITH_API_KEY")

print("=== 🔐 RAG Evaluation System Verification ===")
print(f"Groq API Connection:      {'✅ Connected' if groq_key else '❌ Missing'}")
print(f"LangSmith API Connection: {'✅ Connected' if langsmith_key else '❌ Missing'}")


=== 🔐 RAG Evaluation System Verification ===
Groq API Connection:      ✅ Connected
LangSmith API Connection: ✅ Connected


## 📚 Section 2: Building the In-Memory Knowledge Base & Vector Index

To evaluate a RAG system authentically, we index high-quality technical literature covering three core AI subjects:
1. **LLM-Powered Autonomous Agents**: Planning, decomposition, memory streams, and tool use (from Lilian Weng's foundational research).
2. **Prompt Engineering & In-Context Learning**: Few-shot prompting, chain-of-thought, and systematic biases (majority label, recency bias).
3. **Adversarial Attacks on LLMs**: Token manipulation, gradient attacks, jailbreak prompts, and red-teaming paradigms.

We chunk the documents and store them in an `InMemoryVectorStore` powered by local HuggingFace MiniLM (`all-MiniLM-L6-v2`) 384-dimensional embeddings.


In [2]:
"""
Here, we build the digital library (knowledge base) that our assistant will consult.
We define three rich technical articles about AI agents, prompting strategies, and model security.
Then, we convert each article into dense mathematical vectors using HuggingFace MiniLM embeddings
and store them in an in-memory search index so our bot can retrieve the most relevant facts instantly.
"""

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Curate authoritative domain knowledge
knowledge_sources = [
    Document(
        page_content="""LLM-Powered Autonomous Agents:
In an LLM-powered autonomous agent system, the Large Language Model acts as the central brain.
The agent architecture consists of three essential core components:
1. Planning: The agent breaks down large complex tasks into smaller, manageable subgoals (Task Decomposition) and reflects on past mistakes (Self-Reflection).
2. Memory: Short-term memory utilizes in-context learning, while Long-term memory provides the ability to retain and recall vast information over extended periods via external vector databases.
3. Tool Use: The agent learns to call external APIs for information missing from model weights, including code execution, web search, and calculation.""",
        metadata={"source": "lilian_weng_agents", "topic": "Autonomous Agents"}
    ),
    Document(
        page_content="""Prompt Engineering and Few-Shot Learning Biases:
In-context few-shot learning allows LLMs to adapt to tasks using only a few demonstrative examples in the prompt without parameter updates.
However, research shows few-shot prompting can suffer from three major systemic biases:
1. Majority label bias: Models frequently over-predict whichever answer label appears most frequently in the prompt examples.
2. Recency bias: Models tend to repeat the label or pattern that appeared at the very end of the prompt.
3. Common token bias: Models exhibit a strong preference toward outputting high-frequency vocabulary words over rare technical terms.""",
        metadata={"source": "lilian_weng_prompting", "topic": "Prompt Engineering"}
    ),
    Document(
        page_content="""Adversarial Attacks on Large Language Models:
Adversarial attacks aim to bypass safety guardrails or trick models into generating unintended or hazardous content.
Five primary categories of adversarial attacks are:
1. Token manipulation: Inserting invisible unicode characters or adversarial suffixes that deceive the tokenizer.
2. Gradient-based attacks: Using white-box access to compute token gradients that minimize loss on toxic targets.
3. Jailbreak prompting: Crafting elaborate fictional roleplay or hypothetical scenarios that bypass alignment guardrails.
4. Human red-teaming: Manual probing by security researchers to find unexpected edge-case vulnerabilities.
5. Model red-teaming: Using an automated attacker LLM to iteratively discover weaknesses in a target model.""",
        metadata={"source": "lilian_weng_adversarial", "topic": "AI Safety & Security"}
    )
]

# 2. Split documents into clean, bite-sized passages
text_splitter = RecursiveCharacterTextSplitter(chunk_size=320, chunk_overlap=30)
chunks = text_splitter.split_documents(knowledge_sources)

# 3. Embed chunks with HuggingFace MiniLM and index in memory
print("📦 Generating local embeddings with HuggingFace MiniLM (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

# 4. Create retriever configured to pull top 2 most relevant passages
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"✅ Vector index ready! Successfully indexed {len(chunks)} chunks across {len(knowledge_sources)} documents.")


📦 Generating local embeddings with HuggingFace MiniLM (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3567.50it/s]

✅ Vector index ready! Successfully indexed 11 chunks across 3 documents.


## 🤖 Section 3: The Traceable RAG Pipeline

Next, we construct our complete RAG application. By decorating our function with **`@traceable()`** from the `langsmith` SDK:
- Every execution automatically logs to LangSmith.
- The retrieved context chunks and the generated answer are captured as structured outputs.
- We can inspect the exact documents handed to the LLM during any failed run.


In [3]:
"""
Now we build our complete Question-Answering system (the RAG pipeline).
Notice the '@traceable()' label placed above our function: this acts like a flight data recorder,
automatically logging the user's question, the retrieved documents, and the model's final answer
directly into our LangSmith observability dashboard for complete transparency.
"""

from langsmith import traceable
from langchain_groq import ChatGroq

# Initialize high-performance Groq LLM
generator_llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    groq_api_key=groq_key,
    temperature=0.0,
    max_tokens=150
)

@traceable(name="production_rag_bot")
def rag_bot(question: str) -> dict:
    # 1. Fetch relevant passages from the vector database
    retrieved_docs = retriever.invoke(question)
    context_str = "\n\n".join(f"[Doc {i+1}]: {doc.page_content}" for i, doc in enumerate(retrieved_docs))
    
    # 2. Formulate grounded prompt
    prompt = f"""You are a precise, factual AI research assistant.
Answer the user's question based STRICTLY and ONLY on the provided reference documents.
If the documents do not contain the answer, say "I cannot answer based on the provided context."
Keep your answer clear, informative, and concise (under three sentences).

REFERENCE DOCUMENTS:
{context_str}

USER QUESTION:
{question}

GROUNDED ANSWER:"""

    # 3. Generate response
    response = generator_llm.invoke(prompt)
    answer = response.content.strip()
    
    # Return both the generated text and the retrieved context for evaluation
    return {
        "answer": answer,
        "documents": retrieved_docs
    }

# Test run on sample question
test_q = "What are the core components of an autonomous agent?"
print("=== 🧪 Live Test Run ===")
print("Question:", test_q)
test_out = rag_bot(test_q)
print("-" * 55)
print("Retrieved Passages:", len(test_out["documents"]))
print("Generated Answer:\n", test_out["answer"])


=== 🧪 Live Test Run ===
Question: What are the core components of an autonomous agent?


-------------------------------------------------------
Retrieved Passages: 2
Generated Answer:
 Based on the provided documents, the core components of an LLM-powered autonomous agent include Planning, which involves task decomposition and self-reflection. The text indicates there are three essential core components but only explicitly details Planning.


## 📋 Section 4: Curating the RAG Evaluation Golden Dataset

We define our evaluation dataset in LangSmith containing:
- Specific technical questions testing retrieval precision.
- Ground truth reference answers written by human domain experts.
- Both standard inquiries and adversarial edge cases.


In [4]:
"""
In this step, we create our golden RAG benchmark dataset in LangSmith.
We write out three high-value questions that test our system's understanding of AI agents,
prompt biases, and security attacks, pairing each with the verified ideal answer.
LangSmith will use this dataset to evaluate both retrieval accuracy and answer quality.
"""

from langsmith import Client

client = Client()
rag_dataset_name = "RAG-Triad-Master-Benchmark"

# Clean up older benchmark runs if present
try:
    for existing_ds in client.list_datasets(dataset_name=rag_dataset_name):
        client.delete_dataset(dataset_id=existing_ds.id)
except Exception:
    pass

# Create fresh dataset
rag_dataset = client.create_dataset(
    dataset_name=rag_dataset_name,
    description="Comprehensive evaluation dataset for testing Context Relevance, Faithfulness, and Answer Quality."
)

# Golden RAG Q&A test cases
rag_test_cases = [
    {
        "inputs": {"question": "What three core components make up an LLM-powered autonomous agent?"},
        "outputs": {"answer": "An LLM-powered autonomous agent consists of Planning (subgoal decomposition and self-reflection), Memory (short-term in-context learning and long-term vector storage), and Tool Use (calling external APIs for missing capabilities)."}
    },
    {
        "inputs": {"question": "What three systemic biases can arise during few-shot prompting?"},
        "outputs": {"answer": "The three systemic biases that arise during few-shot prompting are majority label bias (over-predicting frequent labels), recency bias (repeating the final label seen), and common token bias (preferring high-frequency words)."}
    },
    {
        "inputs": {"question": "List five primary categories of adversarial attacks on LLMs."},
        "outputs": {"answer": "The five categories of adversarial attacks are token manipulation, gradient-based attacks, jailbreak prompting, human red-teaming, and model red-teaming."}
    }
]

client.create_examples(
    dataset_id=rag_dataset.id,
    examples=rag_test_cases
)

print(f"✅ RAG Dataset created: '{rag_dataset_name}'")
print(f"📋 Dataset ID: {rag_dataset.id}")
print(f"📊 Golden Examples Count: {len(rag_test_cases)}")


✅ RAG Dataset created: 'RAG-Triad-Master-Benchmark'
📋 Dataset ID: 65cd04c6-0d82-40c2-bd04-5fa5208413e3
📊 Golden Examples Count: 3


## ⚖️ Section 5: The RAG Triad Evaluators

Now we implement the **RAG Triad** using structured LLM-as-a-Judge evaluators:

| Metric | Target Evaluated | Question Asked | Failure Mode Detected |
|:---|:---|:---|:---|
| **1. Retrieval Relevance** | Query vs Retrieved Chunks | Did the retriever fetch passages relevant to the question? | Retrieval Noise / Low Precision |
| **2. Groundedness (Faithfulness)**| Retrieved Chunks vs Answer | Is every claim in the answer backed by the retrieved context? | Hallucination / Untruthfulness |
| **3. Answer Relevance** | Query vs Final Answer | Does the answer directly fulfill the user's original request? | Topic Drift / Evasive Answers |
| **4. Factual Correctness** | Ground Truth vs Final Answer | Does the answer agree with verified reference facts? | Semantic Inaccuracy |


In [5]:
"""
Here, we construct the first two pillars of the RAG Triad:
1. Retrieval Relevance: Our judge checks if the documents pulled by the search engine actually contain
   information needed to answer the question, or if useless clutter was retrieved.
2. Groundedness (Faithfulness): Our judge inspects the final answer to ensure every single claim is
   truthfully backed by the retrieved documents, flagging any imaginary 'hallucinations'.
"""

from typing_extensions import TypedDict, Annotated

# Schema requiring chain-of-thought explanation before awarding a grade
class TriadGrade(TypedDict):
    explanation: Annotated[str, ..., "Step-by-step reasoning explaining the score."]
    score: Annotated[bool, ..., "True if criteria is fully met, False otherwise."]

grader_llm = generator_llm.with_structured_output(TriadGrade)

# 1. Metric: Retrieval Relevance (Context Quality)
def eval_retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    docs_text = "\n\n".join(d.page_content for d in outputs.get("documents", []))
    prompt = f"""You are an expert retrieval judge evaluating a search engine.

USER QUESTION:
{inputs['question']}

RETRIEVED PASSAGES:
{docs_text}

CRITERIA:
- Do the retrieved passages contain relevant keywords, semantic facts, or background directly related to the user question?
- Return score=True if the retrieved passages provide useful information for answering the question.
- Return score=False ONLY if the passages are completely irrelevant or off-topic.
"""
    grade = grader_llm.invoke(prompt)
    return {
        "score": 1 if grade["score"] else 0,
        "comment": grade["explanation"]
    }

# 2. Metric: Groundedness / Faithfulness (Hallucination Detection)
def eval_groundedness(inputs: dict, outputs: dict) -> dict:
    docs_text = "\n\n".join(d.page_content for d in outputs.get("documents", []))
    prompt = f"""You are an eagle-eyed fact-checker evaluating an AI assistant for hallucinations.

SOURCE EVIDENCE (FACTS):
{docs_text}

STUDENT'S GENERATED ANSWER:
{outputs['answer']}

CRITERIA:
- Is every factual claim in the generated answer directly supported by the source evidence?
- Does the answer avoid introducing external assumptions or fabricated statistics not present in the facts?
- Return score=True if the answer is 100% faithful and grounded.
- Return score=False if the answer contains hallucinated or ungrounded claims.
"""
    grade = grader_llm.invoke(prompt)
    return {
        "score": 1 if grade["score"] else 0,
        "comment": grade["explanation"]
    }

print("✅ Pillar 1 (Retrieval Relevance) & Pillar 2 (Groundedness) initialized.")


✅ Pillar 1 (Retrieval Relevance) & Pillar 2 (Groundedness) initialized.


In [6]:
"""
Now, we build the remaining two evaluators:
3. Answer Relevance: Our judge checks if the bot stayed on topic and actually answered the prompt,
   rather than dodging the question or talking about something else.
4. Factual Correctness: Our judge compares the bot's answer to our golden reference answer key to
   verify that all critical facts were communicated accurately.
"""

# 3. Metric: Answer Relevance (Did the bot answer the prompt?)
def eval_answer_relevance(inputs: dict, outputs: dict) -> dict:
    prompt = f"""You are an instructor evaluating whether a student's response is relevant to the question.

QUESTION:
{inputs['question']}

STUDENT'S ANSWER:
{outputs['answer']}

CRITERIA:
- Does the answer directly address the specific question asked by the user?
- Is it concise, focused, and free from evasive topic shifts?
- Return score=True if relevant and helpful; return score=False if irrelevant or off-topic.
"""
    grade = grader_llm.invoke(prompt)
    return {
        "score": 1 if grade["score"] else 0,
        "comment": grade["explanation"]
    }

# 4. Metric: Factual Correctness (Ground Truth Agreement)
def eval_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    prompt = f"""You are a senior examiner comparing a student's answer to the official ground truth.

QUESTION:
{inputs['question']}

OFFICIAL GROUND TRUTH:
{reference_outputs['answer']}

STUDENT'S ANSWER:
{outputs['answer']}

CRITERIA:
- Does the student answer convey the essential factual points stated in the ground truth?
- Minor variations in wording are acceptable as long as facts match.
- Return score=True if factually correct; False otherwise.
"""
    grade = grader_llm.invoke(prompt)
    return {
        "score": 1 if grade["score"] else 0,
        "comment": grade["explanation"]
    }

print("✅ Pillar 3 (Answer Relevance) & Pillar 4 (Factual Correctness) initialized.")


✅ Pillar 3 (Answer Relevance) & Pillar 4 (Factual Correctness) initialized.


## 🚀 Section 6: Running the Full RAG Triad Benchmark

We now orchestrate the end-to-end evaluation using `client.evaluate()`. LangSmith will:
1. Run our traceable RAG pipeline across the entire dataset.
2. Concurrently evaluate all 4 metrics for every question.
3. Collate latencies, pass rates, and individual judge feedback into a consolidated report.


In [7]:
"""
Here is the main event: running our comprehensive RAG Triad examination!
LangSmith will take our test cases, send each through our retriever and generator,
run all four evaluators on each step, and stream the results to our dashboard.
We then convert the experiment data into a pandas table for detailed inspection.
"""

# Wrapper function feeding into LangSmith
def target_rag_app(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

print("🚀 Launching Full RAG Triad Evaluation Benchmark...")
t0 = time.time()

rag_results = client.evaluate(
    target_rag_app,
    data=rag_dataset_name,
    evaluators=[
        eval_retrieval_relevance,
        eval_groundedness,
        eval_answer_relevance,
        eval_correctness
    ],
    experiment_prefix="rag-triad-benchmark",
    metadata={
        "embeddings": "all-MiniLM-L6-v2",
        "llm_model": "qwen/qwen3.8-27b",
        "vector_store": "InMemoryVectorStore",
        "top_k": 2
    }
)

elapsed = time.time() - t0
print(f"✅ RAG Triad Benchmark complete in {elapsed:.2f} seconds!")

# Convert results to DataFrame
df_rag = rag_results.to_pandas()

# Display evaluation scorecard
eval_cols = [
    "inputs.question",
    "outputs.answer",
    "feedback.eval_retrieval_relevance",
    "feedback.eval_groundedness",
    "feedback.eval_answer_relevance",
    "feedback.eval_correctness"
]
df_display = df_rag[[c for c in eval_cols if c in df_rag.columns]]
df_display


🚀 Launching Full RAG Triad Evaluation Benchmark...


View the evaluation results for experiment: 'rag-triad-benchmark-058edd96' at:
https://smith.langchain.com/o/24e3cc69-95f0-469c-bf68-ac531fe7f46f/datasets/65cd04c6-0d82-40c2-bd04-5fa5208413e3/compare?selectedSessions=d1c15117-1e30-4547-bbc7-6501d6295eb7




0it [00:00, ?it/s]

1it [00:02,  2.00s/it]

2it [00:03,  2.00s/it]

3it [00:06,  2.23s/it]

3it [00:06,  2.29s/it]

✅ RAG Triad Benchmark complete in 9.06 seconds!


,inputs.question,outputs.answer,feedback.eval_retrieval_relevance,feedback.eval_groundedness,feedback.eval_answer_relevance,feedback.eval_correctness
0,List five primary categories of adversarial at...,I cannot answer based on the provided context.,1,1,0,0
1,What three systemic biases can arise during fe...,I cannot answer based on the provided context.,1,1,0,0
2,What three core components make up an LLM-powe...,I cannot answer based on the provided context.,1,1,0,0


## 📊 Section 7: Executive RAG Triad Scorecard & Diagnostic Analysis

In production engineering, aggregate metrics help teams diagnose where to invest optimization effort:
- **Low Retrieval Relevance** → Adjust chunk sizes, upgrade embedding model, or implement Re-ranking (Cross-Encoders).
- **Low Groundedness** → Adjust generator temperature (set to 0.0), tighten system prompt constraints, or enforce citation guardrails.
- **Low Answer Relevance** → Re-phrase prompts, implement HyDE (Hypothetical Document Embeddings), or add query decomposition.


In [8]:
"""
Finally, we calculate the overall report card for our RAG system!
We calculate the percentage score for each metric in the RAG Triad plus ground truth accuracy.
This provides a clear diagnostic summary: if retrieval relevance is 100% and groundedness is 100%,
we can deploy our RAG system to production with total confidence that it will never hallucinate!
"""

def metric_pct(col_name):
    if col_name in df_rag.columns:
        return df_rag[col_name].astype(float).mean() * 100
    return 0.0

retrieval_score = metric_pct("feedback.eval_retrieval_relevance")
grounded_score = metric_pct("feedback.eval_groundedness")
relevance_score = metric_pct("feedback.eval_answer_relevance")
correctness_score = metric_pct("feedback.eval_correctness")

scorecard_data = {
    "RAG Triad Metric Dimension": [
        "1. Retrieval Relevance (Context Quality)",
        "2. Groundedness (Zero Hallucination Rate)",
        "3. Answer Relevance (Intent Fulfillment)",
        "4. Factual Correctness (Reference Match)",
        "Overall RAG Production Health"
    ],
    "Score (%)": [
        f"{retrieval_score:.1f}%",
        f"{grounded_score:.1f}%",
        f"{relevance_score:.1f}%",
        f"{correctness_score:.1f}%",
        f"{(retrieval_score + grounded_score + relevance_score + correctness_score) / 4:.1f}%"
    ],
    "Target Benchmark": [
        ">= 90%",
        ">= 95%",
        ">= 90%",
        ">= 90%",
        ">= 92%"
    ],
    "Production Status": [
        "✅ Passed" if retrieval_score >= 90 else "⚠️ Needs Re-ranking",
        "✅ Passed" if grounded_score >= 95 else "❌ High Hallucination Risk",
        "✅ Passed" if relevance_score >= 90 else "⚠️ Prompt Drift",
        "✅ Passed" if correctness_score >= 90 else "⚠️ Accuracy Gap",
        "🟢 Enterprise Ready" if (retrieval_score + grounded_score + relevance_score + correctness_score) / 4 >= 90 else "🔴 Requires Tuning"
    ]
}

scorecard_df = pd.DataFrame(scorecard_data)
print("=========================================================================================")
print("🏆 EXECUTIVE RAG TRIAD PRODUCTION SCORECARD")
print("=========================================================================================")
print(scorecard_df.to_string(index=False))
print("=========================================================================================")


🏆 EXECUTIVE RAG TRIAD PRODUCTION SCORECARD
               RAG Triad Metric Dimension Score (%) Target Benchmark Production Status
 1. Retrieval Relevance (Context Quality)    100.0%           >= 90%          ✅ Passed
2. Groundedness (Zero Hallucination Rate)    100.0%           >= 95%          ✅ Passed
 3. Answer Relevance (Intent Fulfillment)      0.0%           >= 90%   ⚠️ Prompt Drift
 4. Factual Correctness (Reference Match)      0.0%           >= 90%   ⚠️ Accuracy Gap
            Overall RAG Production Health     50.0%           >= 92% 🔴 Requires Tuning


## 🎯 Section 8: Summary & Production Playbook

| RAG Problem | Diagnostic Symptom | Prescribed Solution |
|:---|:---|:---|
| **Retrieval Failure** | Low Retrieval Relevance score | Implement Hybrid Search (BM25 + Dense) or Re-ranking with Cohere/BGE. |
| **Hallucination** | Low Groundedness / Faithfulness score | Tighten system prompt to strictly forbid ungrounded speculation; set `temperature=0.0`. |
| **Query Misalignment** | Low Answer Relevance score | Apply Query Expansion or HyDE (Hypothetical Document Embeddings) before vector search. |
| **Data Drift** | Declining Factual Correctness over time | Schedule periodic re-indexing and update the golden evaluation benchmark in LangSmith. |
